In [8]:
from graph_loader import load_grap
from wl_algorithms import wl_features
from tqdm import tqdm
import pandas as pd

In [3]:
import os
max_nodes = 0
max_edges = 0
for file in tqdm(os.listdir('graphs')):
    if not file.endswith('.graph'):
        continue
    path = os.path.join('graphs', file)
    with open(path) as f:
        g = load_grap(f)
    max_nodes = max(max_nodes, len(g.nodes))
    max_edges = max(max_edges, len(g.edges))

100%|██████████| 301/301 [03:23<00:00,  1.48it/s]


In [4]:
max_nodes, max_edges

(1803537, 3005670)

In [5]:
len(g.edges), len(g.nodes)

(105153, 62069)

In [6]:
f"{max_nodes:,.1f}"

'1,803,537.0'

In [10]:
import json
with open('../parallelise_minizinc/data/results.json') as f:
    data = json.load(f)
with open('./2023_results.json') as f:
    data2023 = json.load(f)

In [11]:
limited = {'chuffed': data['chuffed']['1'] + data2023['chuffed']['1'], 'cp-sat': data['cp-sat']['1'] + data2023['cp-sat']['1']}

In [12]:
limited['chuffed'][0]

{'name': 'cstr_MODEL1507180015',
 'model': 'efm',
 'time': 1200000,
 'objective': None,
 'search': 'Minimise',
 'optimal': 'Unknown',
 'has_solution': False}

In [13]:
reshaped = {}
for v in limited['chuffed']:
    if v['optimal'] == 'Optimal':
        reshaped[(v['model'], v['name'])] = {'chuffed': v['time']}
for v in limited['cp-sat']:
    if v['optimal'] == 'Optimal' and (v['model'], v['name']) in reshaped:
        reshaped[(v['model'], v['name'])]['cp-sat'] = v['time']
    elif v['optimal'] == 'Optimal':
        reshaped[(v['model'], v['name'])] = {'chuffed': 1200000, 'cp-sat': v['time']}
    elif (v['model'], v['name']) in reshaped:
        reshaped[(v['model'], v['name'])]['cp-sat'] = 1200000

In [14]:
len(reshaped)

201

In [15]:
dataset = []
for (m, n), v in reshaped.items():
    if v['chuffed'] // 1000 < v['cp-sat'] // 1000:
        dataset.append({'model': m, 'name': n, 'label':0, 'chuffed': v['chuffed'], 'cp-sat': v['cp-sat']})
    elif v['chuffed'] // 1000 > v['cp-sat'] // 1000:
        dataset.append({'model': m, 'name': n, 'label':1, 'chuffed': v['chuffed'], 'cp-sat': v['cp-sat']})
    elif v['chuffed'] // 1000 == v['cp-sat'] // 1000:
        dataset.append({'model': m, 'name': n, 'label':2, 'chuffed': v['chuffed'], 'cp-sat': v['cp-sat']})
    else:
        raise Exception('what')

In [25]:
dataset[0]

{'model': 'is',
 'name': '1vERFuvviS',
 'label': 2,
 'chuffed': 330,
 'cp-sat': 356,
 'graph': 'graphs/model-sep-1vERFuvviS.graph'}

In [26]:
import pandas as pd
pd.DataFrame(dataset).to_csv('algorithm_selection_dataset.csv')

In [15]:
import os
graphs_names = [os.path.join('graphs', fn) for fn in os.listdir('graphs')]

In [13]:
def where(l, f):
    for _l in l:
        if f(_l):
            return _l

In [14]:
def get_model_name(g):
    n = g.replace('graphs/', '').replace('.graph','')
    if '-sep-' in n:
        [model, name] = n.split('-sep-')
        return model, name
    else:
        return n, n

In [24]:
for d in dataset:
    n = d['name'] if d['name'][-1] != '_' else d['name'][:-1]
    n = n.replace('ExactlyOneSolution_','').replace('base_', '').replace('clear_cp_', '').replace('allocation_only_', '').replace('1id_', '').replace('cstr_', '').replace('queens_mznc2021_', '').replace('opt_', '').replace('.json', '')
    path = where(graphs_names, lambda x: n in x)
    with open(path) as f:
        g = load_grap(f)
    d['graph'] = path
    model, name = get_model_name(path)
    d['name'] = name
    d['model'] = model if model != 'model' else d['model']

In [22]:
d

{'model': 'ttp',
 'name': 'n10_k3_c5000_l10000_u10100_r46',
 'label': 0,
 'chuffed': 359,
 'cp-sat': 1452}

In [27]:
for d in dataset:
    with open(d['graph']) as f:
            g = load_grap(f)
    d['graph'] = g

In [ ]:
l = len(dataset)
train_test = []
buckets = 5
n_bucket = l // buckets
for b in range(buckets):
    test_data = dataset[n_bucket*b: n_bucket *(b+1)]
    train_data = dataset[:n_bucket*b] + dataset[n_bucket *(b+1):]
    train_test.append((train_data, test_data))

In [ ]:
feature_train_test = []
max_iter = 5
for tr, ts in tqdm(train_test):
    train_data = []
    colors = {}
    for t in tr:
        # with open(t['graph']) as f:
        g = t['graph']
        res = wl_features(g, colors, wl_type='node_edge_features', max_iter=max_iter, training=True)
        train_data.append({'res': res, 'label': t['label'], 'chuffed': t['chuffed'], 'cp-sat': t['cp-sat']})
    k = set(int(c) for c in colors.values())
    for t in train_data:
        res = t['res']
        features = [res.count(color) for color in k]
        del t['res']
        t['features'] = features
    test_data = []
    for t in ts:
        # with open(t['graph']) as f:
        g = t['graph']
        res = wl_features(g, colors, wl_type='node_edge_features', max_iter=max_iter, training=False)
        features = [res.count(color) for color in k]
        test_data.append({'features': features, 'label': t['label'], 'chuffed': t['chuffed'], 'cp-sat': t['cp-sat']})
    feature_train_test.append((train_data, test_data))

100%|██████████| 5/5 [03:33<00:00, 42.64s/it]


In [84]:
feature_train_test = [{'train':t[0], 'test':t[1]} for t in feature_train_test]

In [113]:
sum(feature_train_test[0]['train'][0]['features'])

3974

In [61]:
with open('pre-split-3.json', 'w') as f:
    json.dump(feature_train_test, f)

In [85]:
vs = []
for t in feature_train_test:
    v = [0 for _ in t['train'][0]['features']]
    for _t in t['train']:
        for i in range(len(_t['features'])):
            v[i] += _t['features'][i]
    vs.append(v)

In [104]:
rems = []
for v in vs:
    rem = []
    for i, _v in enumerate(v):
        if _v <= 15:
            rem.append(i)
    rems.append(rem)
[len(r) for r in rems]

[2009, 1994, 1884, 1937, 1958]

In [100]:
from copy import deepcopy
pruned = []
for i, d in enumerate(feature_train_test):
    c = deepcopy(d)
    rem = rems[i]
    for r in reversed(rem):
        for e in c['train']:
            e['features'].pop(r)
        for e in c['test']:
            e['features'].pop(r)
    pruned.append(c)

In [47]:
import numpy as np
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import RFECV
from sklearn.metrics import accuracy_score

In [105]:
X_train = np.array([e['features'] for e in pruned[0]['train']])
y_train = np.array([e['label'] for e in pruned[0]['train']])

In [ ]:
clf = RandomForestClassifier(random_state=42)
param_grid = {  #parameters: https://www.researchgate.net/figure/Tested-parameter-grid-for-random-forest-classifier_tbl1_350998771
    'n_estimators': [n for n in range(200,1001, 200)],
    'max_features': ['sqrt', 'log2'],
    'max_depth': [n for n in range(10, 81, 10)] + [None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}
grid_search = GridSearchCV(estimator=clf, param_grid=param_grid, cv=StratifiedKFold(10), scoring='roc_auc_ovr', n_jobs=-1)
grid_search.fit(X_train, y_train)

# Get the best parameters and score
print("Best parameters found: ", grid_search.best_params_)
print("Best cross-validation score: ", grid_search.best_score_)

Best parameters found:  {'max_depth': 20, 'max_features': 'log2', 'min_samples_leaf': 2, 'min_samples_split': 10, 'n_estimators': 200}
Best cross-validation score:  0.7130780423280425


In [106]:
clf = RandomForestClassifier(random_state=42, **grid_search.best_params_)
clf.fit(X_train, y_train)
X_test = np.array([e['features'] for e in pruned[0]['test']])
y_test = np.array([e['label'] for e in pruned[0]['test']])
pred = clf.predict(X_test)
print(accuracy_score(y_test, pred))

0.8


In [107]:
pred_time = 0
chuffed_time = 0
ortools_time = 0
vbs_time = 0
for e in pruned[0]['test']:
    x = np.array([e['features']])
    pred = clf.predict(x)[0]
    if pred == 0 or pred == 2:
        pred_time += e['chuffed']
    elif pred == 1:
        pred_time += e['cp-sat']
    else:
        raise Exception(pred)
    chuffed_time += e['chuffed']
    ortools_time += e['cp-sat']
    vbs_time += min(e['chuffed'], e['cp-sat'])
print(pred_time/vbs_time, pred_time/chuffed_time, pred_time/ortools_time)
print(vbs_time/chuffed_time, vbs_time/ortools_time)

1.0186773732269836 0.8920513687441591 0.34197127183984233
0.8756956738111338 0.33570125422197206


In [11]:
dataset_csv = pd.read_csv('cplex_18.csv')
dataset_csv = dataset_csv[['problem','name','y']]
dataset_csv

,problem,name,y
0,mondoku-gcc-model-balance,10-10-6,1
1,JSP0,12-12-0-1_7,0
2,mondoku-gcc-model-balance,12-12-8,1
3,JSP0,13-14-0-2_6,1
4,JSP0,14-10-0-2_1,1
...,...,...,...
194,group,u12g2pref2,1
195,group,u15g1pref2,0
196,group,u15g3pref0,0
197,group,u15g5pref1,0


In [19]:
dataset = []
for i in range(len(dataset_csv)):
    row = dataset_csv.iloc[i]
    d = {}

    n = row['name'] if row['name'][-1] != '_' else row['name'][:-1]
    n = n.replace('ExactlyOneSolution_','').replace('base_', '').replace('clear_cp_', '').replace('allocation_only_', '').replace('1id_', '').replace('cstr_', '').replace('queens_mznc2021_', '').replace('opt_', '').replace('.json', '')
    path = where(graphs_names, lambda x: n in x)
    with open(path) as f:
        g = load_grap(f)
    d['graph'] = path
    model, name = get_model_name(path)
    d['name'] = name
    d['model'] = model if model != 'model' else row['problem']
    d['y'] = row['y']
    dataset.append(d)

In [ ]:
dataset = pd.DataFrame(dataset)

,graph,name,model,y
0,graphs/mondoku-gcc-model-balance-sep-10-10-6.g...,10-10-6,mondoku-gcc-model-balance,1
1,graphs/JSP0-sep-12-12-0-1_7.graph,12-12-0-1_7,JSP0,0
2,graphs/mondoku-gcc-model-balance-sep-12-12-8.g...,12-12-8,mondoku-gcc-model-balance,1
3,graphs/JSP0-sep-13-14-0-2_6.graph,13-14-0-2_6,JSP0,1
4,graphs/JSP0-sep-14-10-0-2_1.graph,14-10-0-2_1,JSP0,1
...,...,...,...,...
194,graphs/group-sep-u12g2pref2.graph,u12g2pref2,group,1
195,graphs/group-sep-u15g1pref2.graph,u15g1pref2,group,0
196,graphs/group-sep-u15g3pref0.graph,u15g3pref0,group,0
197,graphs/group-sep-u15g5pref1.graph,u15g5pref1,group,0


In [23]:
dataset.to_csv('parallelise_cplex.csv', index=None)